# 注意力机制基础

本notebook介绍注意力机制的核心概念和基础实现。

## 学习目标

- 理解注意力机制的生物学动机
- 掌握查询(Query)、键(Key)、值(Value)的概念
- 学习注意力评分函数
- 实现Nadaraya-Watson核回归作为注意力示例

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

## 1. 注意力机制的动机

### 1.1 生物学中的注意力

**注意力是稀缺资源**:人类的视觉神经系统每秒接收约 $10^8$ 位信息,远超大脑处理能力。

**双组件框架**(威廉·詹姆斯, 1890s):
- **非自主性提示**(Non-volitional cue): 基于环境中物体的突出性和易见性
  - 例子: 红色咖啡杯在黑白报纸中突出,自动吸引注意力
- **自主性提示**(Volitional cue): 基于认知和意识的主观控制
  - 例子: 想读书时,主动将目光聚焦在书上

### 1.2 神经网络中的注意力

**核心问题**: 如何让模型学会"关注"输入的重要部分?

**传统方法的局限**:
- **全连接层**: 所有输入权重相同,无法区分重要性
- **平均池化**: 简单平均,忽略了输入的差异性

**注意力机制的优势**:
- 动态权重: 根据查询和键的相关性动态调整
- 选择性聚合: 只关注与任务相关的输入部分

## 2. 查询、键和值(QKV)

注意力机制的核心是**查询-键-值**三元组:

### 2.1 概念定义

- **查询(Query, Q)**: 自主性提示,表示"我想找什么"
  - 来源: 当前解码器状态、当前时间步的表示
  - 维度: $\mathbf{q} \in \mathbb{R}^{d_q}$

- **键(Key, K)**: 非自主性提示,表示"每个输入的标识"
  - 来源: 编码器的所有隐状态
  - 维度: $\mathbf{k}_i \in \mathbb{R}^{d_k}$, 共 $m$ 个键

- **值(Value, V)**: 感官输入,表示"实际要获取的信息"
  - 来源: 编码器的所有隐状态(通常与键相同)
  - 维度: $\mathbf{v}_i \in \mathbb{R}^{d_v}$, 共 $m$ 个值

### 2.2 注意力汇聚公式

给定查询 $\mathbf{q}$ 和 $m$ 个键-值对 $(\mathbf{k}_1, \mathbf{v}_1), \ldots, (\mathbf{k}_m, \mathbf{v}_m)$:

$$
f(\mathbf{q}, (\mathbf{k}_1, \mathbf{v}_1), \ldots, (\mathbf{k}_m, \mathbf{v}_m)) = \sum_{i=1}^m \alpha(\mathbf{q}, \mathbf{k}_i) \mathbf{v}_i
$$

其中注意力权重 $\alpha(\mathbf{q}, \mathbf{k}_i)$ 通过softmax计算:

$$
\alpha(\mathbf{q}, \mathbf{k}_i) = \text{softmax}(a(\mathbf{q}, \mathbf{k}_i)) = \frac{\exp(a(\mathbf{q}, \mathbf{k}_i))}{\sum_{j=1}^m \exp(a(\mathbf{q}, \mathbf{k}_j))}
$$

**关键**: 注意力评分函数 $a(\mathbf{q}, \mathbf{k}_i)$ 决定了查询和键的相关性!

## 3. 注意力评分函数

评分函数 $a(\mathbf{q}, \mathbf{k})$ 衡量查询和键的相似度。

### 3.1 加性注意力(Additive Attention)

又称**Bahdanau注意力**,用于seq2seq模型:

$$
a(\mathbf{q}, \mathbf{k}) = \mathbf{w}_v^\top \tanh(\mathbf{W}_q \mathbf{q} + \mathbf{W}_k \mathbf{k})
$$

其中:
- $\mathbf{W}_q \in \mathbb{R}^{h \times d_q}$: 查询投影矩阵
- $\mathbf{W}_k \in \mathbb{R}^{h \times d_k}$: 键投影矩阵
- $\mathbf{w}_v \in \mathbb{R}^h$: 输出权重向量
- $h$: 隐藏单元数

**特点**: 可学习参数将不同维度的q和k映射到同一空间

In [ ]:
class AdditiveAttention(nn.Module):
    """加性注意力(Bahdanau注意力)"""
    def __init__(self, key_size, query_size, num_hiddens, dropout=0.1):
        super().__init__()
        self.W_k = nn.Linear(key_size, num_hiddens, bias=False)
        self.W_q = nn.Linear(query_size, num_hiddens, bias=False)
        self.w_v = nn.Linear(num_hiddens, 1, bias=False)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, queries, keys, values, valid_lens=None):
        """
        queries: (batch_size, num_queries, query_size)
        keys: (batch_size, num_keys, key_size)
        values: (batch_size, num_keys, value_size)
        valid_lens: (batch_size,) 或 (batch_size, num_queries)
        """
        # queries: (batch_size, num_queries, num_hiddens)
        queries = self.W_q(queries)
        # keys: (batch_size, num_keys, num_hiddens)
        keys = self.W_k(keys)
        
        # 广播加法: (batch_size, num_queries, 1, num_hiddens) + 
        #            (batch_size, 1, num_keys, num_hiddens)
        features = queries.unsqueeze(2) + keys.unsqueeze(1)
        features = torch.tanh(features)
        
        # scores: (batch_size, num_queries, num_keys)
        scores = self.w_v(features).squeeze(-1)
        
        # 应用掩码和softmax
        attention_weights = self.masked_softmax(scores, valid_lens)
        attention_weights = self.dropout(attention_weights)
        
        # 加权求和: (batch_size, num_queries, value_size)
        return torch.bmm(attention_weights, values)
    
    def masked_softmax(self, X, valid_lens):
        """带掩码的softmax操作"""
        if valid_lens is None:
            return F.softmax(X, dim=-1)
        else:
            shape = X.shape
            if valid_lens.dim() == 1:
                valid_lens = valid_lens.repeat_interleave(shape[1])
            else:
                valid_lens = valid_lens.reshape(-1)
            
            # 将超出有效长度的位置设为很大的负数
            X = X.reshape(-1, shape[-1])
            mask = torch.arange(X.shape[1], device=X.device)[None, :] < valid_lens[:, None]
            X[~mask] = -1e6
            
            return F.softmax(X.reshape(shape), dim=-1)

# 测试加性注意力
batch_size, num_queries, num_keys = 2, 3, 10
query_size, key_size, value_size = 20, 20, 4
num_hiddens = 8

queries = torch.randn(batch_size, num_queries, query_size)
keys = torch.randn(batch_size, num_keys, key_size)
values = torch.randn(batch_size, num_keys, value_size)
valid_lens = torch.tensor([8, 6])  # 每个样本的有效长度

attention = AdditiveAttention(key_size, query_size, num_hiddens)
output = attention(queries, keys, values, valid_lens)

print(f"输出形状: {output.shape}")  # (2, 3, 4)
print(f"输出:\n{output[0, 0, :]}")

### 3.2 缩放点积注意力(Scaled Dot-Product Attention)

**Transformer中使用的评分函数**:

$$
a(\mathbf{q}, \mathbf{k}) = \frac{\mathbf{q}^\top \mathbf{k}}{\sqrt{d}}
$$

其中 $d$ 是查询和键的维度。

**为什么要除以 $\sqrt{d}$?**
- 点积的方差随维度线性增长: $\text{Var}(\mathbf{q}^\top \mathbf{k}) = d$
- 大的点积值会导致softmax梯度消失
- 除以 $\sqrt{d}$ 使方差稳定在1,梯度更稳定

**矩阵形式**(批量计算):

$$
\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d}}\right) \mathbf{V}
$$

其中:
- $\mathbf{Q} \in \mathbb{R}^{n \times d}$: $n$ 个查询
- $\mathbf{K} \in \mathbb{R}^{m \times d}$: $m$ 个键
- $\mathbf{V} \in \mathbb{R}^{m \times v}$: $m$ 个值

**计算复杂度**: $O(n \cdot m \cdot d)$,比加性注意力更高效

In [ ]:
class DotProductAttention(nn.Module):
    """缩放点积注意力"""
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, queries, keys, values, valid_lens=None):
        """
        queries: (batch_size, num_queries, d)
        keys: (batch_size, num_keys, d)
        values: (batch_size, num_keys, value_size)
        """
        d = queries.shape[-1]
        
        # scores: (batch_size, num_queries, num_keys)
        scores = torch.bmm(queries, keys.transpose(1, 2)) / np.sqrt(d)
        
        # 应用掩码和softmax
        attention_weights = self.masked_softmax(scores, valid_lens)
        attention_weights = self.dropout(attention_weights)
        
        # 加权求和: (batch_size, num_queries, value_size)
        return torch.bmm(attention_weights, values)
    
    def masked_softmax(self, X, valid_lens):
        """带掩码的softmax操作"""
        if valid_lens is None:
            return F.softmax(X, dim=-1)
        else:
            shape = X.shape
            if valid_lens.dim() == 1:
                valid_lens = valid_lens.repeat_interleave(shape[1])
            else:
                valid_lens = valid_lens.reshape(-1)
            
            X = X.reshape(-1, shape[-1])
            mask = torch.arange(X.shape[1], device=X.device)[None, :] < valid_lens[:, None]
            X[~mask] = -1e6
            
            return F.softmax(X.reshape(shape), dim=-1)

# 测试点积注意力
batch_size, num_queries, num_keys, d = 2, 3, 10, 8
value_size = 4

queries = torch.randn(batch_size, num_queries, d)
keys = torch.randn(batch_size, num_keys, d)
values = torch.randn(batch_size, num_keys, value_size)
valid_lens = torch.tensor([8, 6])

attention = DotProductAttention()
output = attention(queries, keys, values, valid_lens)

print(f"输出形状: {output.shape}")  # (2, 3, 4)
print(f"输出:\n{output[0, 0, :]}")

## 4. Nadaraya-Watson核回归

用简单的回归问题演示注意力机制!

### 4.1 问题设定

给定训练数据 $\{(x_1, y_1), \ldots, (x_n, y_n)\}$,预测新输入 $x$ 的输出 $\hat{y} = f(x)$。

**生成数据**:
$$
y_i = 2\sin(x_i) + x_i^{0.8} + \epsilon, \quad \epsilon \sim \mathcal{N}(0, 0.5^2)
$$

### 4.2 平均汇聚(基线)

最简单的估计器,忽略输入 $x$:

$$
f(x) = \frac{1}{n}\sum_{i=1}^n y_i
$$

**问题**: 所有预测都相同,性能很差!

In [ ]:
# 生成数据
n_train = 50
x_train = torch.sort(torch.rand(n_train) * 5)[0]

def f(x):
    return 2 * torch.sin(x) + x**0.8

y_train = f(x_train) + torch.randn(n_train) * 0.5

x_test = torch.arange(0, 5, 0.1)
y_truth = f(x_test)
n_test = len(x_test)

# 平均汇聚预测
y_hat_avg = y_train.mean().repeat(n_test)

# 可视化
plt.figure(figsize=(10, 5))
plt.plot(x_test, y_truth, label='Truth', linewidth=2)
plt.plot(x_test, y_hat_avg, label='Average Pooling', linewidth=2)
plt.scatter(x_train, y_train, alpha=0.5, label='Training Data')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.title('平均汇聚: 忽略输入位置')
plt.grid(True, alpha=0.3)
plt.show()

mse_avg = ((y_hat_avg - y_truth)**2).mean()
print(f"平均汇聚MSE: {mse_avg:.4f}")

### 4.3 Nadaraya-Watson核回归

**关键思想**: 根据输入位置 $x$ 对输出 $y_i$ 加权!

$$
f(x) = \sum_{i=1}^n \frac{K(x - x_i)}{\sum_{j=1}^n K(x - x_j)} y_i
$$

其中 $K$ 是核函数。

**注意力视角**:
- 查询: 测试点 $x$
- 键: 训练点 $x_i$
- 值: 训练标签 $y_i$
- 注意力权重: $\alpha(x, x_i) = \frac{K(x - x_i)}{\sum_j K(x - x_j)}$

**高斯核**:
$$
K(u) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{u^2}{2}\right)
$$

**参数化高斯核**(可学习):
$$
K(x - x_i) = \exp\left(-\frac{1}{2}\left(\frac{x - x_i}{\sigma}\right)^2\right)
$$

其中 $\sigma$ 控制注意力范围(带宽)。

In [ ]:
class NWKernelRegression(nn.Module):
    """Nadaraya-Watson核回归(带可学习的注意力权重)"""
    def __init__(self):
        super().__init__()
        # 可学习的带宽参数(对数空间,保证正数)
        self.log_sigma = nn.Parameter(torch.tensor(0.0))
        
    def forward(self, x_train, y_train, x_test):
        """
        x_train: (n_train,)
        y_train: (n_train,)
        x_test: (n_test,)
        """
        sigma = torch.exp(self.log_sigma)
        
        # 计算距离: (n_test, n_train)
        distances = x_test[:, None] - x_train[None, :]
        
        # 高斯核: (n_test, n_train)
        attention_scores = torch.exp(-0.5 * (distances / sigma)**2)
        
        # 归一化为注意力权重
        attention_weights = attention_scores / attention_scores.sum(dim=1, keepdim=True)
        
        # 加权求和: (n_test,)
        y_pred = (attention_weights * y_train[None, :]).sum(dim=1)
        
        return y_pred, attention_weights

# 训练模型
model = NWKernelRegression()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

for epoch in range(100):
    y_pred, _ = model(x_train, y_train, x_test)
    loss = F.mse_loss(y_pred, y_truth)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Sigma: {torch.exp(model.log_sigma).item():.4f}")

# 最终预测
with torch.no_grad():
    y_hat_nw, attention_weights = model(x_train, y_train, x_test)

# 可视化结果
plt.figure(figsize=(12, 5))

# 预测曲线
plt.subplot(1, 2, 1)
plt.plot(x_test, y_truth, label='Truth', linewidth=2)
plt.plot(x_test, y_hat_nw, label='NW Kernel Regression', linewidth=2)
plt.scatter(x_train, y_train, alpha=0.5, label='Training Data')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.title('Nadaraya-Watson核回归')
plt.grid(True, alpha=0.3)

# 注意力权重热力图
plt.subplot(1, 2, 2)
plt.imshow(attention_weights.numpy(), aspect='auto', cmap='hot', interpolation='nearest')
plt.colorbar(label='Attention Weight')
plt.xlabel('Training Sample Index')
plt.ylabel('Test Sample Index')
plt.title('注意力权重可视化')

plt.tight_layout()
plt.show()

mse_nw = ((y_hat_nw - y_truth)**2).mean()
print(f"\nNW核回归MSE: {mse_nw:.4f}")
print(f"相比平均汇聚提升: {(1 - mse_nw/mse_avg)*100:.2f}%")

### 4.4 注意力权重分析

从热力图可以看出:
- **对角线模式**: 测试点关注附近的训练点
- **带宽效应**: $\sigma$ 小时,关注范围窄(局部);$\sigma$ 大时,关注范围广(全局)
- **自适应权重**: 不同测试点的注意力分布不同,体现了注意力机制的灵活性

In [ ]:
# 可视化特定测试点的注意力分布
test_indices = [10, 25, 40]  # 选择3个测试点

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, test_idx in enumerate(test_indices):
    ax = axes[i]
    
    # 注意力权重条形图
    ax.bar(range(n_train), attention_weights[test_idx].numpy(), alpha=0.6)
    ax.axvline(x=test_idx * n_train / n_test, color='red', linestyle='--', 
               label=f'Test Point x={x_test[test_idx]:.2f}')
    ax.set_xlabel('Training Sample Index')
    ax.set_ylabel('Attention Weight')
    ax.set_title(f'Test Point {test_idx}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 小结

### 核心概念

1. **注意力机制 = 查询-键-值三元组**
   - 查询(Q): "我想找什么"
   - 键(K): "每个输入的标识"
   - 值(V): "实际要获取的信息"

2. **注意力汇聚 = 值的加权和**
   $$
   \text{Output} = \sum_{i} \alpha(q, k_i) v_i
   $$

3. **评分函数决定权重**
   - 加性注意力: $a(q, k) = w_v^\top \tanh(W_q q + W_k k)$
   - 点积注意力: $a(q, k) = \frac{q^\top k}{\sqrt{d}}$

### Nadaraya-Watson核回归的启示

- **非参数方法**: 不假设函数形式,直接从数据学习
- **注意力视角**: 核函数 $\Leftrightarrow$ 注意力评分函数
- **可学习权重**: 通过反向传播优化带宽参数

### 下一步

- **Bahdanau注意力**: 将注意力应用于seq2seq模型
- **多头注意力**: 捕获多种依赖关系
- **自注意力**: 序列内部的注意力机制
- **Transformer**: 完全基于注意力的架构

## 练习

1. **评分函数对比**: 将Nadaraya-Watson的高斯核替换为其他核函数(如三角核、Epanechnikov核),观察性能差异。

2. **带宽选择**: 尝试不同的初始 $\sigma$ 值,观察收敛速度和最终性能。

3. **多维扩展**: 将NW核回归推广到多维输入 $\mathbf{x} \in \mathbb{R}^d$。

4. **加性 vs 点积**: 在同一任务上比较加性注意力和点积注意力的性能和计算效率。

5. **掩码效果**: 在masked_softmax中尝试不同的掩码值(如-1e6, -1e9, -inf),观察数值稳定性。